# MemoryBench on Azure ML: dataset walkthrough

This is a self-contained notebook walkthrough. It does **not** call a CLI, scripts, or hidden helper package.

Set Azure and dataset values in `.env`, then run the cells top-to-bottom. The notebook uses Azure SDKs directly so you can see exactly what each step does.

Currently, `DATASET=longmemeval` is supported end to end. `DATASET=locomo` is adapted into the same MemoryBench QA format. `DATASET=statebench` prepares/registers train trajectories, while official task execution still uses the STATE-Bench runner.


## 0. Find the repo root and import SDK dependencies

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import shutil
import subprocess
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any
from urllib.parse import parse_qs, quote, unquote, urlparse
from urllib.request import Request, urlopen

from azure.ai.ml import Input, MLClient, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data, Environment
from azure.core.exceptions import HttpResponseError
from azure.identity import AzureCliCredential
from azure.storage.blob import BlobServiceClient

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise RuntimeError("Start VS Code/Jupyter with the membench-aml repo as the working folder.")

print(f"repo={ROOT}")

## 1. Load `.env` and choose run settings

For the normal path, users only edit `.env`. The constants below provide safe defaults and can be overridden with environment variables.

In [ ]:
def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ[key] = value


def env_str(name: str, default: str = "") -> str:
    return os.environ.get(name, default).strip()


def env_path(name: str, default: str) -> Path:
    value = Path(env_str(name, default))
    return value if value.is_absolute() else ROOT / value


load_dotenv(ROOT / ".env")

DATASET_PRESETS = {
    "longmemeval": {
        "benchmark_id": "longmemeval-s",
        "asset_name": "longmemeval-s",
        "uri": "hf://xiaowu0162/longmemeval-cleaned/longmemeval_s_cleaned.json?revision=main",
        "source_path": r"data\sources\longmemeval_s_cleaned.json",
        "output_dir": r"data\longmemeval-s",
    },
    "locomo": {
        "benchmark_id": "locomo-qa",
        "asset_name": "locomo10",
        "uri": "https://raw.githubusercontent.com/snap-research/locomo/main/data/locomo10.json",
        "source_path": r"data\sources\locomo10.json",
        "output_dir": r"data\locomo10",
    },
    "statebench": {
        "benchmark_id": "statebench-agent-learning",
        "asset_name": "statebench-agent-learning",
        "uri": "https://github.com/microsoft/STATE-Bench.git",
        "source_path": r"external\STATE-Bench",
        "output_dir": r"data\statebench-agent-learning",
    },
}

DATASET = env_str("DATASET", "longmemeval").lower()
if DATASET not in DATASET_PRESETS:
    raise ValueError(f"Unknown DATASET={DATASET!r}. Use one of: {', '.join(DATASET_PRESETS)}")

DATASET_PERCENT_TEXT = env_str("DATASET_PERCENT", "1")
try:
    DATASET_PERCENT = float(DATASET_PERCENT_TEXT)
except ValueError as exc:
    raise ValueError("DATASET_PERCENT must be a number from 1 to 100") from exc
if DATASET_PERCENT <= 0 or DATASET_PERCENT > 100:
    raise ValueError("DATASET_PERCENT must be greater than 0 and less than or equal to 100")

preset = DATASET_PRESETS[DATASET]
DATASET_BENCHMARK_ID = preset["benchmark_id"]
DATASET_NAME = f"{preset['asset_name']}-p{int(DATASET_PERCENT) if DATASET_PERCENT.is_integer() else str(DATASET_PERCENT).replace('.', '_')}"
DATASET_VERSION = env_str("DATASET_VERSION", "1")
DATASET_URI = env_str("DATASET_URI") or preset["uri"]
SOURCE_PATH = ROOT / preset["source_path"]
PREPARED_DATASET_DIR = ROOT / f"{preset['output_dir']}-p{int(DATASET_PERCENT) if DATASET_PERCENT.is_integer() else str(DATASET_PERCENT).replace('.', '_')}"
SEED = 42

# Fixed notebook defaults. Change these in the cell only if you know you need to.

# Foundry Memory retrieval budget: pass up to this many retrieved memories to the answer model.
MAX_MEMORIES = 8

# AML experiment/run name. Includes dataset, k, and percent so runs are easy to identify in Azure ML.
EXPERIMENT_NAME = f"{DATASET}-k{MAX_MEMORIES}-p{int(DATASET_PERCENT) if DATASET_PERCENT.is_integer() else str(DATASET_PERCENT).replace('.', '_')}"

# LongMemEval history mode. "full" means each case keeps its full haystack/history.
HISTORY_MODE = "full"

# Ingest memory once per case and reuse the same retrieved memories for normal vs structured answer prompts.
REUSE_MEMORY = True

# Use "all" for a full run. To rerun only memory failures, set QA_CONDITIONS to "foundry-memory,foundry-memory-structured".
QA_CONDITIONS = env_str("QA_CONDITIONS", "all")

# Notebook/demo default: keep going if one session fails ingestion, but record the incomplete-ingestion details.
# Set True only for stricter benchmark-validity runs where partial memory should invalidate that case.
FAIL_ON_PARTIAL_INGESTION = False

# Retry/backoff controls for Foundry Memory ingestion. These are intentionally conservative because ingestion can hit transient service errors.
MEMORY_INGEST_RETRIES = 10
MEMORY_INGEST_RATE_LIMIT_DELAY_SECONDS = 90
MEMORY_INGEST_RETRY_DELAY_SECONDS = 20
MEMORY_INGEST_SESSION_DELAY_SECONDS = 10

# Foundry Memory option defaults for this smoke harness. User-profile/procedural extraction currently fails intermittently in service-side model auth.
MEMORY_CHAT_SUMMARY_ENABLED = True
MEMORY_USER_PROFILE_ENABLED = False
MEMORY_PROCEDURAL_ENABLED = False

# Model-call throttling for low-TPM deployments. The runner also retries 429s with backoff.
LLM_CALL_DELAY_SECONDS = 5
LLM_RATE_LIMIT_DELAY_SECONDS = 90

# How long to wait for extracted memories to appear after ingestion before answering.
MEMORY_WAIT_TIMEOUT_SECONDS = 600

# Reusable Azure ML environment for command jobs. The notebook registers this once, then jobs reuse it.
AML_ENVIRONMENT_NAME = "memorybench-runner"
AML_ENVIRONMENT_VERSION = env_str("AML_ENVIRONMENT_VERSION", "2")
AML_ENVIRONMENT_CONDA = ROOT / "infra" / "memorybench-runner-conda.yml"

# Base Azure ML image for the command job environment.
ENVIRONMENT_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04"

# Submit the repo root so AML has the vendored QA runner.
QA_CODE_PATH = ROOT

AGGREGATE_OUTPUT = ROOT / "outputs" / "aggregate.json"
AGGREGATE_CONTAINER = "azureml"

# MemoryBench QA runner settings. Required for DATASET=longmemeval and DATASET=locomo.
# ANSWER_DEPLOYMENT is the model-under-test for QA answers and STATE-Bench agent calls.
ANSWER_DEPLOYMENT = env_str("ANSWER_DEPLOYMENT")
JUDGE_DEPLOYMENT = env_str("JUDGE_DEPLOYMENT")
QA_ENV = {
    "AZURE_AI_PROJECT_ENDPOINT": env_str("AZURE_AI_PROJECT_ENDPOINT"),
    "AZURE_OPENAI_ENDPOINT": env_str("AZURE_OPENAI_ENDPOINT"),
    "ANSWER_DEPLOYMENT": ANSWER_DEPLOYMENT,
    "JUDGE_DEPLOYMENT": JUDGE_DEPLOYMENT,
    "EMBEDDING_DEPLOYMENT": env_str("EMBEDDING_DEPLOYMENT"),
    "MEMORY_STORE_NAME": env_str("MEMORY_STORE_NAME", "memorybench_live_qa"),
}

# STATE-Bench reuses the answer deployment for the agent and the judge deployment for evaluation.
STATE_BENCH_EVAL_ENDPOINT = env_str("STATE_BENCH_EVAL_ENDPOINT", env_str("AZURE_OPENAI_ENDPOINT"))
STATE_BENCH_EVAL_DEPLOYMENTS = env_str("STATE_BENCH_EVAL_DEPLOYMENTS", JUDGE_DEPLOYMENT)
STATE_BENCH_AGENT_ENDPOINT = env_str("STATE_BENCH_AGENT_ENDPOINT", env_str("AZURE_OPENAI_ENDPOINT"))
STATE_BENCH_AGENT_DEPLOYMENTS = env_str("STATE_BENCH_AGENT_DEPLOYMENTS", ANSWER_DEPLOYMENT)
STATE_BENCH_AGENT_MODEL_NAME = env_str("STATE_BENCH_AGENT_MODEL_NAME", "statebench-agent")
STATE_BENCH_NUM_RUNS = 1
STATE_BENCH_NUM_WORKERS = 1

print(f"dataset={DATASET} percent={DATASET_PERCENT:g}%")
print("source", SOURCE_PATH)
print("prepared", PREPARED_DATASET_DIR)
print("asset", f"azureml:{DATASET_NAME}:{DATASET_VERSION}")

## 2. Create Azure SDK clients and run preflight checks

In [ ]:
subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID", "")
resource_group = os.environ.get("AZURE_RESOURCE_GROUP", "")
workspace_name = os.environ.get("AZURE_ML_WORKSPACE", "")
compute_name = os.environ.get("AZURE_ML_COMPUTE", "")
storage_account = os.environ.get("AZURE_STORAGE_ACCOUNT", "")
foundry_project_endpoint = os.environ.get("AZURE_AI_PROJECT_ENDPOINT", "")
identity_client_id = os.environ.get("AZURE_CLIENT_ID", "")

required = {
    "AZURE_SUBSCRIPTION_ID": subscription_id,
    "AZURE_RESOURCE_GROUP": resource_group,
    "AZURE_ML_WORKSPACE": workspace_name,
    "AZURE_ML_COMPUTE": compute_name,
    "AZURE_STORAGE_ACCOUNT": storage_account,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise RuntimeError("Missing required .env values: " + ", ".join(missing))

# This walkthrough uses Azure CLI auth intentionally.
# If this cell fails, run `az login` in a terminal, then rerun the cell.
credential = AzureCliCredential(process_timeout=60)
try:
    credential.get_token("https://management.azure.com/.default")
except Exception as exc:
    raise RuntimeError("Azure CLI authentication failed. Run `az login`, then rerun this cell.") from exc

ml_client = MLClient(credential, subscription_id, resource_group, workspace_name)
blob_service = BlobServiceClient(
    account_url=f"https://{storage_account}.blob.core.windows.net",
    credential=credential,
)

checks: list[tuple[str, str, str]] = []

try:
    workspace = ml_client.workspaces.get(workspace_name)
    checks.append(("passed", "aml_workspace", f"workspace={workspace.name}"))
except Exception as exc:
    checks.append(("failed", "aml_workspace", f"{type(exc).__name__}: {exc}"))

try:
    compute = ml_client.compute.get(compute_name)
    checks.append(("passed", "aml_compute", f"compute={compute.name}"))
except Exception as exc:
    checks.append(("failed", "aml_compute", f"{type(exc).__name__}: {exc}"))

try:
    first_page = next(blob_service.list_containers(results_per_page=1).by_page())
    checks.append(("passed", "blob_storage", f"visible_containers={len(list(first_page))}"))
except Exception as exc:
    checks.append((
        "failed",
        "blob_storage",
        f"{type(exc).__name__}: {exc}. Check blob data-plane RBAC and storage public network access.",
    ))

if foundry_project_endpoint:
    try:
        from azure.ai.projects import AIProjectClient

        project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=credential)
        stores = list(project_client.beta.memory_stores.list())
        checks.append(("passed", "foundry_project", f"memory_stores={len(stores)}"))
    except Exception as exc:
        checks.append(("failed", "foundry_project", f"{type(exc).__name__}: {exc}"))
else:
    checks.append(("warning", "foundry_project", "AZURE_AI_PROJECT_ENDPOINT is not set."))

for status, name, message in checks:
    print(f"{status.upper()}\t{name}\t{message}")

if any(status == "failed" for status, _, _ in checks):
    raise RuntimeError("Preflight failed. Fix failed checks before continuing.")

## 3. Fetch the benchmark source file

This cell uses the built-in source for the selected dataset. For private or gated datasets, download/copy the source file into the local source path printed by the settings cell, then skip this cell.

In [ ]:
def resolve_dataset_source(source: str) -> str:
    parsed = urlparse(source.strip())
    if parsed.scheme in {"http", "https"}:
        return source
    if parsed.scheme != "hf":
        raise ValueError("DATASET_URI must be http(s), git https, or hf://")

    query = parse_qs(parsed.query)
    revision = (query.get("revision") or query.get("rev") or ["main"])[0] or "main"
    parts = []
    if parsed.netloc:
        parts.append(unquote(parsed.netloc))
    parts.extend(unquote(part) for part in parsed.path.strip("/").split("/") if part)
    if len(parts) < 2:
        raise ValueError("hf:// source must include a dataset id and file path")
    dataset_id = "/".join(parts[:2]) if len(parts) >= 3 else parts[0]
    file_path = "/".join(parts[2:]) if len(parts) >= 3 else parts[1]
    encoded_dataset_id = "/".join(quote(part, safe="") for part in dataset_id.split("/"))
    encoded_path = "/".join(quote(part, safe="") for part in file_path.split("/"))
    encoded_revision = quote(revision, safe="")
    return f"https://huggingface.co/datasets/{encoded_dataset_id}/resolve/{encoded_revision}/{encoded_path}"


if DATASET == "statebench":
    if SOURCE_PATH.exists():
        print("STATE-Bench repo already exists at:", SOURCE_PATH)
    else:
        SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("Cloning STATE-Bench repo to:", SOURCE_PATH)
        subprocess.run(["git", "-c", "core.autocrlf=false", "clone", DATASET_URI, str(SOURCE_PATH)], check=True)
        subprocess.run(["git", "-C", str(SOURCE_PATH), "config", "core.autocrlf", "false"], check=True)
elif not DATASET_URI:
    print("No built-in source configured. Expecting local file at:", SOURCE_PATH)
else:
    resolved_url = resolve_dataset_source(DATASET_URI)
    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temp_path = SOURCE_PATH.with_name(SOURCE_PATH.name + ".part")
    request = Request(resolved_url, headers={"User-Agent": "membench-aml-notebook/0.1"})
    with urlopen(request, timeout=60) as response, temp_path.open("wb") as destination:
        shutil.copyfileobj(response, destination)
    temp_path.replace(SOURCE_PATH)
    print(f"downloaded={SOURCE_PATH}")
    print(f"bytes={SOURCE_PATH.stat().st_size}")

## 4. Prepare a sampled benchmark folder

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def pct_count(total: int, percent: float) -> int:
    return max(1, round(total * percent / 100)) if total else 0


def case_id(case: dict[str, Any]) -> str:
    value = case.get("question_id")
    if value is None:
        raise ValueError("case is missing question_id")
    return str(value)


def locomo_session_number(evidence_id: str) -> str:
    # Evidence looks like D8:9. The D number maps to session_8.
    return evidence_id.split(":", 1)[0].lstrip("D")


if DATASET == "longmemeval":
    if not SOURCE_PATH.exists():
        raise RuntimeError(f"Dataset source file not found: {SOURCE_PATH}. Run the fetch cell, or download/copy the file yourself.")

    cases = json.loads(SOURCE_PATH.read_text(encoding="utf-8"))
    if not isinstance(cases, list):
        raise ValueError(f"LongMemEval source must be a JSON list: {SOURCE_PATH}")

    selected = list(cases)
    random.Random(SEED).shuffle(selected)
    selected = selected[:pct_count(len(cases), DATASET_PERCENT)]
    selected = sorted(selected, key=case_id)
    case_file_name = "longmemeval-s-raw.json"
elif DATASET == "locomo":
    if not SOURCE_PATH.exists():
        raise RuntimeError(f"LoCoMo source file not found: {SOURCE_PATH}. Run the fetch cell first.")

    samples = json.loads(SOURCE_PATH.read_text(encoding="utf-8"))
    if not isinstance(samples, list):
        raise ValueError(f"LoCoMo source must be a JSON list: {SOURCE_PATH}")

    locomo_cases = []
    for sample in samples:
        conversation = sample["conversation"]
        session_numbers = sorted(
            int(key.split("_", 1)[1])
            for key in conversation
            if key.startswith("session_") and not key.endswith("_date_time")
        )
        sessions = []
        for number in session_numbers:
            session_key = f"session_{number}"
            date_key = f"session_{number}_date_time"
            turns = [
                {"role": "user" if index % 2 == 0 else "assistant", "content": f"{turn.get('speaker')}: {turn.get('text', '')}"}
                for index, turn in enumerate(conversation.get(session_key, []))
            ]
            sessions.append({"id": str(number), "date": str(conversation.get(date_key, "")), "turns": turns})

        for qa_index, qa in enumerate(sample.get("qa", []), start=1):
            answer_session_ids = sorted({locomo_session_number(evidence) for evidence in qa.get("evidence", []) if evidence})
            category = qa.get("category")
            question_type = "temporal-reasoning" if category == 2 else "multi-session" if category == 3 else "single-session-user"
            locomo_cases.append(
                {
                    "question_id": f"{sample['sample_id']}-{qa_index}",
                    "question_type": question_type,
                    "question": qa["question"],
                    "answer": str(qa.get("answer", qa.get("adversarial_answer", ""))),
                    "question_date": sessions[-1]["date"] if sessions else "",
                    "haystack_session_ids": [session["id"] for session in sessions],
                    "haystack_dates": [session["date"] for session in sessions],
                    "haystack_sessions": [session["turns"] for session in sessions],
                    "answer_session_ids": answer_session_ids,
                }
            )

    random.Random(SEED).shuffle(locomo_cases)
    selected = locomo_cases[:pct_count(len(locomo_cases), DATASET_PERCENT)]
    selected = sorted(selected, key=case_id)
    case_file_name = "locomo-qa-raw.json"
elif DATASET == "statebench":
    if not SOURCE_PATH.exists():
        raise RuntimeError(f"STATE-Bench repo not found: {SOURCE_PATH}. Run the fetch cell first.")

    trajectory_root = SOURCE_PATH / "datasets" / "train_task_trajectories"
    files = sorted(trajectory_root.glob("*/*.json"))
    if not files:
        raise RuntimeError(f"No STATE-Bench train trajectories found under {trajectory_root}")
    random.Random(SEED).shuffle(files)
    selected_files = sorted(files[:pct_count(len(files), DATASET_PERCENT)])
    selected = []
    STATEBENCH_TASKS_BY_DOMAIN = defaultdict(list)
    for file in selected_files:
        rel = file.relative_to(SOURCE_PATH).as_posix()
        domain = file.parent.name
        task_id = file.stem
        payload = json.loads(file.read_text(encoding="utf-8"))
        selected.append({"path": rel, "domain": domain, "task_id": task_id, "trajectory": payload})
        STATEBENCH_TASKS_BY_DOMAIN[domain].append(task_id)
    case_file_name = "statebench-train-trajectories.json"
else:
    raise ValueError(f"Unsupported DATASET={DATASET!r}")

SELECTED_CASE_COUNT = len(selected)
CASE_LIMIT = SELECTED_CASE_COUNT
SHARD_COUNT = max(1, (SELECTED_CASE_COUNT + 4) // 5)
dataset_hash = sha256_json(selected)
source_sha256 = sha256_file(SOURCE_PATH) if SOURCE_PATH.is_file() else dataset_hash
output_dir = PREPARED_DATASET_DIR
output_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "schemaVersion": "0.1.0",
    "benchmark": DATASET,
    "benchmarkId": DATASET_BENCHMARK_ID,
    "benchmarkSet": DATASET_NAME,
    "version": str(DATASET_VERSION),
    "objective": "memory-qa-retrieval" if DATASET in {"longmemeval", "locomo"} else "agent-learning-trajectories",
    "caseCount": SELECTED_CASE_COUNT,
    "caseIds": [case_id(case) if "question_id" in case else case["path"] for case in selected],
    "datasetHash": dataset_hash,
    "sourceSha256": source_sha256,
    "source": str(SOURCE_PATH),
    "sample": {"samplePercent": DATASET_PERCENT, "selectedCaseCount": SELECTED_CASE_COUNT, "seed": SEED},
}

(output_dir / "benchmark-set.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
case_payload = json.dumps(selected, indent=2, ensure_ascii=False)
(output_dir / case_file_name).write_text(case_payload, encoding="utf-8")

# The MemoryBench QA runner expects this canonical case filename for QA datasets.
if DATASET in {"longmemeval", "locomo"}:
    (output_dir / "longmemeval-s-raw.json").write_text(case_payload, encoding="utf-8")
    (output_dir / "longmemeval-s-20-raw.json").write_text(case_payload, encoding="utf-8")

print(f"cases={SELECTED_CASE_COUNT}")
print(f"AML jobs={SHARD_COUNT} (derived as 1 job per 5 cases)")
print(f"datasetHash={dataset_hash}")
print(f"outputDir={output_dir}")
if DATASET == "statebench":
    print("STATE-Bench data is prepared/registerable as train trajectories. Submitting official STATE-Bench tasks requires the STATE-Bench runner, not the LongMemEval QA runner.")

## 5. Register the benchmark folder as an Azure ML data asset

In [ ]:
asset_name = DATASET_NAME
asset_version = str(DATASET_VERSION)

data_asset = Data(
    name=asset_name,
    version=asset_version,
    type=AssetTypes.URI_FOLDER,
    path=str(output_dir),
    description=f"MemoryBench LongMemEval-S benchmark set {asset_name}:{asset_version}",
    tags={
        "memorybench.benchmark_id": DATASET_BENCHMARK_ID,
        "memorybench.benchmark_set": DATASET_NAME,
        "memorybench.dataset_hash": dataset_hash,
        "memorybench.case_count": str(len(selected)),
        "memorybench.objective": "memory-qa-retrieval",
    },
)

try:
    registered = ml_client.data.create_or_update(data_asset)
except HttpResponseError as exc:
    # AML data asset versions are immutable. If this exact version exists, reuse it.
    if "already exists" not in str(exc) and "DataVersionPropertyImmutable" not in str(exc):
        raise
    registered = ml_client.data.get(name=asset_name, version=asset_version)

print(f"asset=azureml:{registered.name}:{registered.version}")
print(registered.id)

## 6. Register the Azure ML job environment

This creates or reuses a named Azure ML environment with the benchmark runner dependencies. Jobs reference this environment instead of installing packages inside every run.

In [ ]:
if not AML_ENVIRONMENT_CONDA.exists():
    raise RuntimeError(f"Missing AML environment file: {AML_ENVIRONMENT_CONDA}")

runner_environment = Environment(
    name=AML_ENVIRONMENT_NAME,
    version=AML_ENVIRONMENT_VERSION,
    image=ENVIRONMENT_IMAGE,
    conda_file=str(AML_ENVIRONMENT_CONDA),
    description="MemoryBench AML runner environment with Azure SDKs and pinned STATE-Bench dependency.",
)

try:
    registered_environment = ml_client.environments.create_or_update(runner_environment)
except HttpResponseError as exc:
    # AML environment versions are immutable. If this version exists, reuse it.
    if "already exists" not in str(exc) and "immutable" not in str(exc).lower():
        raise
    registered_environment = ml_client.environments.get(name=AML_ENVIRONMENT_NAME, version=AML_ENVIRONMENT_VERSION)
AML_ENVIRONMENT_REF = f"azureml:{registered_environment.name}:{registered_environment.version}"
print(f"environment={AML_ENVIRONMENT_REF}")

## 7. Load the experiment config and dry-run AML jobs

In [ ]:
if DATASET == "statebench":
    resolved_statebench_env = {
        "STATE_BENCH_EVAL_ENDPOINT": STATE_BENCH_EVAL_ENDPOINT,
        "STATE_BENCH_EVAL_DEPLOYMENTS": STATE_BENCH_EVAL_DEPLOYMENTS,
        "STATE_BENCH_AGENT_ENDPOINT": STATE_BENCH_AGENT_ENDPOINT,
        "STATE_BENCH_AGENT_DEPLOYMENTS": STATE_BENCH_AGENT_DEPLOYMENTS,
        "STATE_BENCH_AGENT_MODEL_NAME": STATE_BENCH_AGENT_MODEL_NAME,
    }
    missing_statebench_env = [name for name, value in resolved_statebench_env.items() if not value]
    if missing_statebench_env:
        raise RuntimeError(
            "DATASET=statebench needs these .env values before submitting: " + ", ".join(missing_statebench_env)
        )

    statebench_env = {
        key: value
        for key, value in os.environ.items()
        if key.startswith("STATE_BENCH_") or key in {"AZURE_CLIENT_ID", "OPENAI_API_KEY"}
    }
    statebench_env.update(resolved_statebench_env)
    jobs = []
    for domain, task_ids in sorted(STATEBENCH_TASKS_BY_DOMAIN.items()):
        task_csv = ",".join(sorted(task_ids))
        display_name = f"{EXPERIMENT_NAME}-{domain}"
        statebench_output_dir = f"outputs/statebench/{domain}"
        statebench_command = (
            "export PYTHONPATH=. && "
            "python -m state_bench.scripts.run_batch "
            f"--domain {domain} "
            f"--tasks {task_csv} "
            f"--agent-model-name {STATE_BENCH_AGENT_MODEL_NAME} "
            f"--num-runs {STATE_BENCH_NUM_RUNS} "
            f"--num-workers {STATE_BENCH_NUM_WORKERS} "
            f"--output-dir {statebench_output_dir} && "
            "python -m state_bench.scripts.compute_metrics "
            f"--domain {domain} "
            f"--results-dir {statebench_output_dir} "
            f"--num-runs {STATE_BENCH_NUM_RUNS} "
            "--ignore-missing-runs "
            f"--output-dir {statebench_output_dir}"
        )
        job = command(
            display_name=display_name,
            experiment_name=EXPERIMENT_NAME,
            description=f"STATE-Bench {domain} smoke for {len(task_ids)} task(s)",
            code=str(SOURCE_PATH),
            command=statebench_command,
            environment=AML_ENVIRONMENT_REF,
            compute=compute_name,
            environment_variables=statebench_env,
            tags={
                "memorybench.benchmark_id": DATASET_BENCHMARK_ID,
                "memorybench.dataset": DATASET,
                "memorybench.experiment": EXPERIMENT_NAME,
                "memorybench.domain": domain,
                "memorybench.task_count": str(len(task_ids)),
            },
        )
        jobs.append(job)
else:
    benchmark = {"id": DATASET_BENCHMARK_ID, "dataset": f"azureml:{DATASET_NAME}:{DATASET_VERSION}"}
    print(f"Using dataset asset: {benchmark['dataset']}")
    experiment_name = EXPERIMENT_NAME
    max_memories = MAX_MEMORIES
    runner_code_path = QA_CODE_PATH
    runner_command = (
        "PYTHONPATH=runner/benchmark-runner/src MEMORYBENCH_BENCHMARK_SET_DIR=${{inputs.benchmark_set}} "
        "python -m memorybench_worker.memory_qa_live"
    )

    jobs = []
    for shard_index in range(SHARD_COUNT):
        display_name = f"{experiment_name}-k{max_memories}-shard-{shard_index}of{SHARD_COUNT}"
        env_vars = {
            "MEMORYBENCH_OUTPUT_DIR": "./outputs",
            "MEMORYBENCH_LONGMEMEVAL_CASE_LIMIT": str(CASE_LIMIT),
            "MEMORYBENCH_LONGMEMEVAL_HISTORY_MODE": HISTORY_MODE,
            "MEMORYBENCH_MEMORY_MAX_MEMORIES": str(max_memories),
            "MEMORYBENCH_MEMORY_SCOPE_PREFIX": f"{experiment_name}-k{max_memories}",
            "MEMORYBENCH_MEMORY_INGEST_RETRIES": str(MEMORY_INGEST_RETRIES),
            "MEMORYBENCH_MEMORY_INGEST_RETRY_DELAY_SECONDS": str(MEMORY_INGEST_RETRY_DELAY_SECONDS),
            "MEMORYBENCH_MEMORY_INGEST_RATE_LIMIT_DELAY_SECONDS": str(MEMORY_INGEST_RATE_LIMIT_DELAY_SECONDS),
            "MEMORYBENCH_MEMORY_INGEST_SESSION_DELAY_SECONDS": str(MEMORY_INGEST_SESSION_DELAY_SECONDS),
            "MEMORYBENCH_MEMORY_WAIT_TIMEOUT_SECONDS": str(MEMORY_WAIT_TIMEOUT_SECONDS),
            "MEMORYBENCH_MEMORY_CHAT_SUMMARY_ENABLED": "1" if MEMORY_CHAT_SUMMARY_ENABLED else "0",
            "MEMORYBENCH_MEMORY_USER_PROFILE_ENABLED": "1" if MEMORY_USER_PROFILE_ENABLED else "0",
            "MEMORYBENCH_MEMORY_PROCEDURAL_ENABLED": "1" if MEMORY_PROCEDURAL_ENABLED else "0",
            "MEMORYBENCH_CONDITIONS": QA_CONDITIONS,
            "MEMORYBENCH_REUSE_MEMORY": "1" if REUSE_MEMORY else "0",
            "MEMORYBENCH_FAIL_ON_PARTIAL_MEMORY_INGESTION": "1" if FAIL_ON_PARTIAL_INGESTION else "0",
            "MEMORYBENCH_LLM_CALL_DELAY_SECONDS": str(LLM_CALL_DELAY_SECONDS),
            "MEMORYBENCH_LLM_RATE_LIMIT_DELAY_SECONDS": str(LLM_RATE_LIMIT_DELAY_SECONDS),
            "MEMORYBENCH_SHARD_INDEX": str(shard_index),
            "MEMORYBENCH_SHARD_COUNT": str(SHARD_COUNT),
        }
        env_vars.update(QA_ENV)
        missing_qa_env = [
            key
            for key in ("AZURE_AI_PROJECT_ENDPOINT", "AZURE_OPENAI_ENDPOINT", "ANSWER_DEPLOYMENT", "JUDGE_DEPLOYMENT", "EMBEDDING_DEPLOYMENT")
            if not env_vars.get(key)
        ]
        if missing_qa_env:
            raise RuntimeError("DATASET=" + DATASET + " needs these .env values before submitting: " + ", ".join(missing_qa_env))
        if identity_client_id:
            env_vars["AZURE_CLIENT_ID"] = identity_client_id

        job = command(
            display_name=display_name,
            experiment_name=experiment_name,
            description=f"MemoryBench {experiment_name} shard {shard_index}/{SHARD_COUNT}",
            code=str(runner_code_path),
            command=runner_command,
            environment=AML_ENVIRONMENT_REF,
            compute=compute_name,
            inputs={"benchmark_set": Input(type="uri_folder", path=benchmark["dataset"])},
            environment_variables=env_vars,
            tags={
                "memorybench.benchmark_id": benchmark["id"],
                "memorybench.dataset": DATASET,
                "memorybench.experiment": experiment_name,
                "memorybench.memory_system": "foundry-memory",
                "memorybench.max_memories": str(max_memories),
                "memorybench.shard_index": str(shard_index),
                "memorybench.shard_count": str(SHARD_COUNT),
            },
        )
        jobs.append(job)

for index, job in enumerate(jobs):
    print(f"dry_run-{index}\t{job.display_name}")

## 8. Submit AML jobs

This cell is gated so a dry run cannot accidentally submit real jobs. With `SUBMIT_REAL_JOBS = False`, no AML jobs are created and the next status/wait/aggregate cells have nothing to find.

Change `SUBMIT_REAL_JOBS` to `True` only when you are ready to submit real AML jobs.

In [ ]:
SUBMIT_REAL_JOBS = True

submitted_job_names: list[str] = []
if not SUBMIT_REAL_JOBS:
    print("No AML jobs submitted. Set SUBMIT_REAL_JOBS = True, then rerun this cell when ready.")
else:
    for job in jobs:
        result = ml_client.jobs.create_or_update(job)
        submitted_job_names.append(str(result.name))
        print(f"{getattr(result, 'status', 'submitted')}\t{result.name}\t{result.display_name}")

    print("\nSubmitted job names:")
    for name in submitted_job_names:
        print(f"  - {name}")

## 9. Check AML job status

Run this after submitting real AML jobs. If the submit cell left `SUBMIT_REAL_JOBS = False`, this cell should say no jobs were found.

In [ ]:
rows = []
for job in ml_client.jobs.list():
    display_name = str(getattr(job, "display_name", "") or "")
    tags = getattr(job, "tags", {}) or {}
    if tags.get("memorybench.experiment") == EXPERIMENT_NAME or display_name.startswith(EXPERIMENT_NAME):
        rows.append((str(getattr(job, "status", "")), str(job.name), display_name))

if not rows:
    print("No jobs found for this experiment. If SUBMIT_REAL_JOBS was False, this is expected.")
    print("To continue: set SUBMIT_REAL_JOBS = True in the submit cell, rerun it, then rerun this status cell.")
else:
    for status, name, display_name in sorted(rows, key=lambda item: item[2]):
        print(f"{status}\t{name}\t{display_name}")

## 10. Wait for AML jobs to finish

Azure ML SDK can wait for jobs for us with `ml_client.jobs.stream(job_name)`. Run this cell after real jobs have been submitted. It streams each matching job until completion and keeps the completed job names for aggregation.

In [ ]:
stream_rows = rows or []
if not stream_rows:
    for job in ml_client.jobs.list():
        display_name = str(getattr(job, "display_name", "") or "")
        tags = getattr(job, "tags", {}) or {}
        if tags.get("memorybench.experiment") == EXPERIMENT_NAME or display_name.startswith(EXPERIMENT_NAME):
            stream_rows.append((str(getattr(job, "status", "")), str(job.name), display_name))

if not stream_rows:
    print("No jobs found yet. If SUBMIT_REAL_JOBS was False, submit real jobs first.")
else:
    job_names_to_stream = [name for _, name, _ in sorted(stream_rows, key=lambda item: item[2])]
    for job_name in job_names_to_stream:
        print(f"\nStreaming AML job until completion: {job_name}")
        ml_client.jobs.stream(job_name)

    completed_job_names = job_names_to_stream
    print("\nStreaming complete. These jobs are ready for aggregation:")
    for job_name in completed_job_names:
        print(f"  - {job_name}")

## 11. Aggregate results

After jobs complete, run this cell. It uses the jobs from the submit/wait cells in the same notebook session.

If you have not submitted real jobs yet, skip this cell.

In [ ]:
CONDITION_TO_SUMMARY_KEY = {
    "question-only": "baseline",
    "full-transcript": "selectedTranscript",
    "oracle-context": "oracleContext",
    "foundry-memory-structured": "foundryMemoryStructured",
    "foundry-memory": "foundryMemory",
}


def condition_from_case_blob(blob_name: str) -> str | None:
    filename = blob_name.rsplit("/", 1)[-1]
    if not filename.endswith(".json"):
        return None
    for condition in CONDITION_TO_SUMMARY_KEY:
        if filename.startswith(f"{condition}-"):
            return condition
    if filename.startswith("selected-transcript-"):
        return "full-transcript"
    return None


def metric_row(name: str, rows: list[dict[str, Any]]) -> dict[str, Any]:
    correct = sum(1 for row in rows if row.get("answer_correct"))
    total = len(rows)
    clean = [row for row in rows if not row.get("errors") and row.get("memory_ingestion_complete", True)]
    clean_correct = sum(1 for row in clean if row.get("answer_correct"))
    return {
        "condition": name,
        "correct": correct,
        "total": total,
        "accuracy": correct / total if total else 0.0,
        "percent_correct": (correct / total * 100) if total else 0.0,
        "clean_correct": clean_correct,
        "clean_total": len(clean),
        "clean_percent_correct": (clean_correct / len(clean) * 100) if clean else 0.0,
        "error_count": sum(len(row.get("errors") or []) for row in rows),
    }


job_names = (
    list(globals().get("completed_job_names", []))
    or list(globals().get("submitted_job_names", []))
    or [str(job.name) for job in globals().get("jobs", []) if getattr(job, "name", None)]
)
if not job_names:
    print("No submitted jobs found in this notebook session, so there is nothing to aggregate yet.")
    print("Submit real AML jobs, wait for them to complete, then rerun this cell.")
elif DATASET == "statebench":
    container = blob_service.get_container_client(AGGREGATE_CONTAINER)
    found_metrics = False
    for job_name in job_names:
        print(f"\nSTATE-Bench results for {job_name}:")
        prefix = f"ExperimentRun/dcid.{job_name}/outputs/statebench/"
        for blob in container.list_blobs(name_starts_with=prefix):
            if not blob.name.endswith("metrics.json"):
                continue
            found_metrics = True
            payload = container.download_blob(blob.name).readall().decode("utf-8")
            print(blob.name)
            print(payload[:4000])
    if not found_metrics:
        print("No STATE-Bench metrics.json artifacts found yet. Check that the job completed after the metrics step.")
else:
    container_name = AGGREGATE_CONTAINER
    rows_by_condition: dict[str, list[dict[str, Any]]] = defaultdict(list)
    container = blob_service.get_container_client(container_name)

    for job_name in job_names:
        per_job_counts: dict[str, int] = defaultdict(int)
        prefix = f"ExperimentRun/dcid.{job_name}/outputs/cases/"
        for blob in container.list_blobs(name_starts_with=prefix):
            condition = condition_from_case_blob(blob.name)
            if condition is None:
                continue
            payload = container.download_blob(blob.name).readall()
            rows_by_condition[condition].append(json.loads(payload.decode("utf-8")))
            per_job_counts[condition] += 1

    metrics_by_condition = {
        summary_key: metric_row(summary_key, rows_by_condition.get(condition, []))
        for condition, summary_key in CONDITION_TO_SUMMARY_KEY.items()
    }
    foundry_acc = metrics_by_condition["foundryMemory"]["accuracy"]
    structured_acc = metrics_by_condition["foundryMemoryStructured"]["accuracy"]
    baseline_acc = metrics_by_condition["baseline"]["accuracy"]
    memory_rows = rows_by_condition.get("foundry-memory", [])

    aggregate_payload = {
        "schemaVersion": "0.1.0",
        "source": "aml-case-blobs",
        "jobNames": job_names,
        "conditions": {
            "question_only": metrics_by_condition["baseline"],
            "full_transcript": metrics_by_condition["selectedTranscript"],
            "selected_transcript": metrics_by_condition["selectedTranscript"],
            "oracle_context": metrics_by_condition["oracleContext"],
            "foundry_memory": metrics_by_condition["foundryMemory"],
            "foundry_memory_structured": metrics_by_condition["foundryMemoryStructured"],
        },
        "derived": {
            "answer_accuracy_lift_pp": (foundry_acc - baseline_acc) * 100,
            "structured_answer_accuracy_lift_pp": (structured_acc - baseline_acc) * 100,
            "structured_vs_unstructured_delta_pp": (structured_acc - foundry_acc) * 100,
            "memory_ingestion_incomplete_count": sum(
                1 for row in memory_rows if not row.get("memory_ingestion_complete", True)
            ),
        },
    }
    primary_metrics = {
        "answer_correct": metrics_by_condition["foundryMemory"]["correct"],
        "answer_total": metrics_by_condition["foundryMemory"]["total"],
        "answer_accuracy": metrics_by_condition["foundryMemory"]["accuracy"],
        "answer_percent_correct": metrics_by_condition["foundryMemory"]["percent_correct"],
        "baseline_percent_correct": metrics_by_condition["baseline"]["percent_correct"],
        "structured_answer_percent_correct": metrics_by_condition["foundryMemoryStructured"]["percent_correct"],
        "full_transcript_percent_correct": metrics_by_condition["selectedTranscript"]["percent_correct"],
        "selected_transcript_percent_correct": metrics_by_condition["selectedTranscript"]["percent_correct"],
        "oracle_context_percent_correct": metrics_by_condition["oracleContext"]["percent_correct"],
        "memory_lift_pp": (foundry_acc - baseline_acc) * 100,
    }

    output_path = AGGREGATE_OUTPUT
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(aggregate_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    primary_path = output_path.with_name(output_path.stem + "-primary-metrics.json")
    primary_path.write_text(json.dumps(primary_metrics, indent=2, ensure_ascii=False), encoding="utf-8")

    print(json.dumps(primary_metrics, indent=2))
    print(f"aggregate={output_path}")
    print(f"primary_metrics={primary_path}")